In [0]:
storage_account = "sadataproject01"

# Paste your Storage Account Key (from Azure Portal -> Storage Accounts -> sadataproject01 -> Access keys)
storage_account_key = "<YOUR_AZURE_STORAGE_ACCOUNT_KEY>"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Customer SCD Type 2 Dimension (`silver.dim_customers_scd2`)
# MAGIC 
# MAGIC - **Source**: `bronze.user_addr_scd2` (Change Feed) & `bronze.users` (Profile Metadata)
# MAGIC - **Target**: `silver.dim_customers_scd2` (Delta Table with SCD2 tracking)
# MAGIC - **Key Mechanics**:
# MAGIC   - Tracks address movements over time using windowed chronological sequencing.
# MAGIC   - Derives effective intervals (`valid_from`, `valid_to`, `is_current`).
# MAGIC   - Creates a deterministic surrogate key (`customer_sk`) via SHA-256.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Storage paths & environment configuration
storage_account = "sadataproject01"
silver_customers_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/dim_customers_scd2/"

# Ensure catalog schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# COMMAND ----------

# 1. Read source data from Bronze
df_addr = spark.table("bronze.user_addr_scd2")
df_users = spark.table("bronze.users") if spark.catalog.tableExists("bronze.users") else None

# COMMAND ----------

# 2. Build SCD Type 2 timeline per customer
window_spec = Window.partitionBy("usr_id").orderBy(F.col("usr_addr_change_date").asc())

df_scd2_prep = (
    df_addr
    .filter(F.col("usr_id").isNotNull() & F.col("usr_addr_change_date").isNotNull())
    .dropDuplicates(["usr_id", "usr_addr_change_date"])
    .withColumn("valid_from", F.to_date(F.col("usr_addr_change_date")))
    .withColumn(
        "next_change_date",
        F.lead("usr_addr_change_date").over(window_spec)
    )
    .withColumn(
        "valid_to",
        F.coalesce(
            F.date_sub(F.to_date(F.col("next_change_date")), 1),
            F.to_date(F.lit("9999-12-31"))
        )
    )
    .withColumn(
        "is_current",
        F.when(F.col("valid_to") == F.to_date(F.lit("9999-12-31")), F.lit(True)).otherwise(F.lit(False))
    )
    .drop("next_change_date")
)

# COMMAND ----------

# 3. Generate deterministic Surrogate Key (SHA-256 of usr_id + valid_from)
df_scd2_with_sk = df_scd2_prep.withColumn(
    "customer_sk",
    F.sha2(
        F.concat_ws("||", F.col("usr_id").cast("string"), F.col("valid_from").cast("string")),
        256
    )
)

# COMMAND ----------

# 4. Optional enrichment with base profile information
if df_users is not None:
    audit_cols = [c for c in ["_ingested_at", "_source_file", "_batch_id", "_row_hash"] if c in df_users.columns]
    df_users_clean = df_users.drop(*audit_cols).dropDuplicates(["usr_id"])
    
    df_dim_customers = (
        df_scd2_with_sk.join(df_users_clean, on="usr_id", how="left")
        .withColumn("_transformed_at", F.current_timestamp())
    )
else:
    df_dim_customers = df_scd2_with_sk.withColumn("_transformed_at", F.current_timestamp())



In [0]:
# COMMAND ----------

# 5. Save directly as a managed Delta table in Unity Catalog
(
    df_dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_customers_scd2")
)

# 6. Optimize table layout for fast analytical lookups
spark.sql("OPTIMIZE silver.dim_customers_scd2 ZORDER BY (usr_id, is_current)")

# COMMAND ----------

# 7. Validate SCD Type 2 output
display(
    spark.table("silver.dim_customers_scd2")
    .select("customer_sk", "usr_id", "usr_city", "usr_state", "valid_from", "valid_to", "is_current")
    .orderBy("usr_id", "valid_from")
    .limit(10)
)